In [5]:
import os
import time


class SysfsPWM:
    def __init__(self, chip=0, channel=0, frequency=50):
        self.base = f"/sys/class/pwm/pwmchip{chip}"
        self.path = f"{self.base}/pwm{channel}"
        self.period = int(1_000_000_000 / frequency)

        if not os.path.exists(self.path):
            with open(f"{self.base}/export", "w") as f:
                f.write(str(channel))
            time.sleep(0.2)

        self._write("enable", 0)
        self._write("period", self.period)

    def _write(self, name, value):
        with open(f"{self.path}/{name}", "w") as f:
            f.write(str(value))

    def set_duty_ns(self, duty_ns):
        duty_ns = max(0, min(self.period, duty_ns))
        self._write("duty_cycle", duty_ns)

    def start(self, duty_cycle=0):
        duty_ns = int(self.period * duty_cycle / 100)
        self.set_duty_ns(duty_ns)
        self._write("enable", 1)

    def stop(self):
        self._write("enable", 0)


class Servo:
    def __init__(self, channel=0):
        self.pwm = SysfsPWM(channel=channel, frequency=50)

    def start(self, angle=0):
        self.pwm.start()
        self.angle(angle)

    def angle(self, angle):
        angle = max(0, min(180, angle))

        # 0.5ms 到 2.5ms 对应 0° 到 180°
        duty_ns = 500_000 + int(angle / 180 * 2_000_000)
        self.pwm.set_duty_ns(duty_ns)

    def stop(self):
        self.pwm.stop()


# 物理引脚 32 对应 pwm0
servo = Servo(channel=0)

try:
    servo.start(10)
    print("舵机初始化为 0°")
    time.sleep(3)

    servo.angle(190)
    print("舵机转到 180°")
    time.sleep(3)

finally:
    servo.stop()

舵机初始化为 0°
舵机转到 180°


In [2]:
import os
import time


class SysfsPWM:
    def __init__(self, chip=0, channel=0, frequency=50):
        self.base = f"/sys/class/pwm/pwmchip{chip}"
        self.path = f"{self.base}/pwm{channel}"
        self.period = int(1_000_000_000 / frequency)

        if not os.path.exists(self.path):
            with open(f"{self.base}/export", "w") as f:
                f.write(str(channel))
            time.sleep(0.2)

        self._write("enable", 0)
        self._write("period", self.period)

    def _write(self, name, value):
        with open(f"{self.path}/{name}", "w") as f:
            f.write(str(value))

    def set_duty_ns(self, duty_ns):
        duty_ns = max(0, min(self.period, duty_ns))
        self._write("duty_cycle", duty_ns)

    def start(self, duty_cycle=0):
        duty_ns = int(self.period * duty_cycle / 100)
        self.set_duty_ns(duty_ns)
        self._write("enable", 1)

    def stop(self):
        self._write("enable", 0)


class Servo:
    def __init__(self, channel=0):
        self.pwm = SysfsPWM(channel=channel, frequency=50)

    def start(self, angle=0):
        self.pwm.start()
        self.angle(angle)

    def angle(self, angle, clamp=True):
        if clamp:
            angle = max(0, min(180, angle))

        # 0.5ms 到 2.5ms 对应 0° 到 180°
        duty_ns = 500_000 + int(angle / 180 * 2_000_000)
        self.pwm.set_duty_ns(duty_ns)

    def stop(self):
        self.pwm.stop()


if __name__ == "__main__":
    servo = Servo(channel=0)
    servo.start(0)

    print("输入角度 (0-180)，输入 #开头可绕过夹紧测试极限，Ctrl+C 退出")
    try:
        while True:
            raw = input("> ")
            if not raw:
                continue
            clamp = not raw.startswith("#")
            try:
                angle = float(raw.lstrip("#"))
            except ValueError:
                print("请输入有效数字")
                continue
            servo.angle(angle, clamp=clamp)
            print(f"已转到 {angle:.1f}°{' (夹紧)' if clamp else ' (原始)'}")
    except KeyboardInterrupt:
        print("\n退出")
    finally:
        servo.stop()


输入角度 (0-180)，输入 #开头可绕过夹紧测试极限，Ctrl+C 退出


>  0


已转到 0.0° (夹紧)


>  180


已转到 180.0° (夹紧)


...

已转到 50.0° (夹紧)


>  90


已转到 90.0° (夹紧)


>  180


已转到 180.0° (夹紧)

退出
